In [1]:
import pandas as pd

In [2]:
sp500_df = pd.read_csv("../Data/Raw_data/SP500_raw_data.csv", skiprows=3, names=["Date", "Close", "High", "Low", "Open", "Volume"], index_col="Date") #sets up s&p500 data frame - the intial csv file had a weird format with sub headings for each column name so this fixes that
sp500_df.index = pd.to_datetime(sp500_df.index) #makes the date column a date data type
cpi_df = pd.read_csv("../Data/Raw_data/cpi_raw_data.csv", index_col="Date") #sets up CPI data frame
cpi_df.index = pd.to_datetime(cpi_df.index)
inflation_expectations_df = pd.read_csv("../Data/Raw_data/inflation_expectations_raw_data.csv", index_col="Date") #creates a data frame for inflation expectations for each date
inflation_expectations_df.index = pd.to_datetime(inflation_expectations_df.index)

In [3]:
cpi_df = cpi_df.loc["1998-12-31":"2025-12-31"] #gets CPI values for these years
print(cpi_df[cpi_df['CPI'].isna()]) #outputs any rows with missing values
cpi_df.interpolate(inplace=True) #Fills missing values with predicted values
cpi_df.sort_values(by="Date", inplace=True)
cpi_df["YOY_inflation"] = (cpi_df.CPI - cpi_df.CPI.shift(12)) / (cpi_df.CPI.shift(12)) * 100   #creates a new column for inflation as a %
cpi_df["Inflation_volatility"] = cpi_df.YOY_inflation.rolling(12).std()
cpi_df = cpi_df.loc["2000-11-30":"2025-12-31"] #Limits data frame dates further since I needed extra dates before to calculate the year on year inflation
cpi_df["Date_month"] = cpi_df.index.to_period("M") #Converts dates to monthly values to make merging data sets later on easier
cpi_df



            CPI
Date           
2025-10-01  NaN


,CPI,YOY_inflation,Inflation_volatility,Date_month
Date,,,,
2000-12-01,174.600,3.436019,0.286482,2000-12
2001-01-01,175.600,3.721205,0.239086,2001-01
2001-02-01,176.000,3.529412,0.229091,2001-02
2001-03-01,176.100,2.982456,0.248253,2001-03
2001-04-01,176.400,3.218256,0.224944,2001-04
...,...,...,...,...
2025-08-01,323.291,2.938592,0.232003,2025-08
2025-09-01,324.245,3.022572,0.242695,2025-09
2025-10-01,324.654,2.858718,0.243175,2025-10


In [4]:
sp500_df = sp500_df[["Close"]] #keeps the only nescesary column for my analysis
sp500_df = sp500_df.groupby(sp500_df.index.to_period("M")).head(1) #Returns the first record for each month only - data is converted from daily to monthly
sp500_df["SP500_monthly_return"] = (sp500_df.Close - sp500_df.Close.shift(1)) / (sp500_df.Close.shift(1)) * 100 #calculates return as a %
sp500_df["SP500_volatility"] = sp500_df.SP500_monthly_return.rolling(12).std()
sp500_df = sp500_df.loc["2000-11-30":"2025-12-31"]
sp500_df["Date_month"] = sp500_df.index.to_period("M") #converts data to monthly format to make merging easier
sp500_df

,Close,SP500_monthly_return,SP500_volatility,Date_month
Date,,,,
2000-12-01,1315.229980,-7.457677,4.760707,2000-12
2001-01-02,1283.270020,-2.429990,4.567131,2001-01
2001-02-01,1373.469971,7.028914,5.041273,2001-02
2001-03-01,1241.229980,-9.628168,5.732928,2001-03
2001-04-02,1145.869995,-7.682701,5.115852,2001-04
...,...,...,...,...
2025-08-01,6238.009766,0.645368,3.235395,2025-08
2025-09-02,6415.540039,2.845944,3.270414,2025-09
2025-10-01,6711.200195,4.608500,3.366205,2025-10


In [5]:
inflation_expectations_df = inflation_expectations_df.loc["2000-11-30":"2025-12-31"] #reduces size of data set
print(inflation_expectations_df[inflation_expectations_df['Inflation_expectations'].isna()]) #prints records with missing values
inflation_expectations_df.sort_values(by="Date", inplace=True)
inflation_expectations_df["Date_month"] = inflation_expectations_df.index.to_period("M") #Converts dates to monthly format to make merging easier
inflation_expectations_df



Empty DataFrame
Columns: [Inflation_expectations]
Index: []


,Inflation_expectations,Date_month
Date,,
2000-12-01,2.8,2000-12
2001-01-01,3.0,2001-01
2001-02-01,2.8,2001-02
2001-03-01,2.8,2001-03
2001-04-01,3.1,2001-04
...,...,...
2025-08-01,4.8,2025-08
2025-09-01,4.7,2025-09
2025-10-01,4.6,2025-10


In [6]:
sp500_df.rename(columns={"Close":"SP500_close"}, inplace=True)
sp500_cpi_df = pd.merge(sp500_df, cpi_df, how="outer", on="Date_month") #merges my s&p500 df and my cpi df
df = pd.merge(sp500_cpi_df, inflation_expectations_df, how="outer", on="Date_month") #creates my combined df containing all data needed for my analysis
df["Inflation_surprise"] = df.YOY_inflation - df.Inflation_expectations #calculates inflation surprise (the difference between actual inflation and expected inflation)
df = df.set_index("Date_month") #sets my date column to the index
df.info()
df = df[["SP500_monthly_return", "SP500_volatility", "YOY_inflation", "Inflation_volatility", "Inflation_surprise"]] #gets rid of unescesary columns
df["Inflation_lag"] = df.YOY_inflation.shift(1) #Decided to add a one month delay to all my stock market variables
df["Volatility_lag"] = df.Inflation_volatility.shift(1)
df["Surprise_lag"] = df.Inflation_surprise.shift(1)
df.dropna(inplace=True)
df
df.to_csv("../Data/Cleaned_data/Clean.csv")

<class 'pandas.DataFrame'>
PeriodIndex: 301 entries, 2000-12 to 2025-12
Freq: M
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SP500_close             301 non-null    float64
 1   SP500_monthly_return    301 non-null    float64
 2   SP500_volatility        301 non-null    float64
 3   CPI                     301 non-null    float64
 4   YOY_inflation           301 non-null    float64
 5   Inflation_volatility    301 non-null    float64
 6   Inflation_expectations  301 non-null    float64
 7   Inflation_surprise      301 non-null    float64
dtypes: float64(8)
memory usage: 21.2 KB
